# Extractor de Tablas — Presupuesto Participativo CDMX

Este notebook permite explorar y probar la extracción de datos desde fotografías de tablas del Presupuesto Participativo de la Ciudad de México.

**Flujo:**
1. Configurar API key de Anthropic
2. Cargar una imagen de ejemplo
3. Extraer datos con Claude Vision
4. Validar colonias contra catálogo oficial
5. Exportar resultados a CSV


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Image as IPImage

# Asegurarse de que el módulo src sea importable
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.extractor import TableExtractor

print("Módulos cargados correctamente.")

## 1. Configuración

Asegúrate de tener tu API key de Anthropic configurada. Puedes crear un archivo `.env` en la raíz del proyecto:

```
ANTHROPIC_API_KEY=sk-ant-...
```

O configurarla directamente:

In [ ]:
# Opcional: si no tienes .env, descomenta y agrega tu key aquí
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

extractor = TableExtractor()
print("Extractor inicializado.")

## 2. Explorar imágenes de ejemplo

Coloca tus imágenes en `images/examples/` y ejecuta la celda siguiente para listarlas.

In [ ]:
IMAGES_DIR = ROOT / "images" / "examples"
extensions = {".jpg", ".jpeg", ".png", ".webp"}

imagenes = [p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in extensions]

if imagenes:
    print(f"Imágenes encontradas ({len(imagenes)}):")
    for img in sorted(imagenes):
        print(f"  {img.name}")
else:
    print("No hay imágenes en images/examples/")
    print("Agrega fotografías de tablas del PP para continuar.")

## 3. Previsualizar una imagen

In [ ]:
# Cambia el nombre de la imagen que quieres probar
IMAGEN_PRUEBA = "mi_tabla.jpg"  # <-- EDITA ESTE NOMBRE

img_path = IMAGES_DIR / IMAGEN_PRUEBA

if img_path.exists():
    display(IPImage(filename=str(img_path), width=700))
else:
    print(f"Imagen no encontrada: {img_path}")
    print("Agrega la imagen o cambia IMAGEN_PRUEBA en la celda anterior.")

## 4. Extraer datos de la tabla

In [ ]:
if img_path.exists():
    print(f"Extrayendo datos de: {IMAGEN_PRUEBA}")
    df = extractor.extract(str(img_path))
    
    print(f"\n=== Metadatos detectados ===")
    print(f"  Alcaldía : {df.attrs.get('alcaldia', 'No detectada')}")
    print(f"  Colonia  : {df.attrs.get('colonia', 'No detectada')}")
    print(f"  Año      : {df.attrs.get('anio', 'No detectado')}")
    print(f"  Notas    : {df.attrs.get('notas', 'Ninguna')}")
    
    print(f"\n=== Datos extraídos ({len(df)} filas) ===")
    display(df)
else:
    print("Agrega una imagen de prueba primero.")

## 5. Validar colonia contra catálogo oficial

In [ ]:
# Cargar catálogo de colonias
colonias_path = ROOT / "data" / "colonias" / "colonias_cdmx.csv"
colonias_df = pd.read_csv(colonias_path)

print(f"Catálogo cargado: {len(colonias_df)} colonias en {colonias_df['alcaldia'].nunique()} alcaldías\n")
print(colonias_df.groupby("alcaldia").size().to_string())

In [ ]:
# Validar la colonia detectada en la imagen (si se extrajo alguna)
if 'df' in dir() and df.attrs.get('colonia'):
    colonia_detectada = df.attrs['colonia']
    alcaldia_detectada = df.attrs.get('alcaldia')
    
    resultado = extractor.validate_colonia(colonia_detectada, alcaldia_detectada)
    
    if resultado['encontrado']:
        print(f"✓ '{colonia_detectada}' encontrada en el catálogo:")
        for c in resultado['coincidencias']:
            print(f"  - {c['colonia']} / {c['alcaldia']}")
    else:
        print(f"✗ '{colonia_detectada}' NO encontrada en el catálogo.")
        print("  Verifica si hay errores de OCR o el nombre exacto en el catálogo.")

# Búsqueda manual
BUSCAR_COLONIA = "Tepito"  # <-- CAMBIA ESTE VALOR
resultado_manual = extractor.validate_colonia(BUSCAR_COLONIA)
print(f"\nBúsqueda manual — '{BUSCAR_COLONIA}':")
print(f"  Encontrada: {resultado_manual['encontrado']}")
for c in resultado_manual['coincidencias']:
    print(f"  - {c['colonia']} / {c['alcaldia']}")

## 6. Procesar múltiples imágenes en lote

In [ ]:
# Descomenta para procesar todas las imágenes en images/examples/
# df_consolidado = extractor.process_directory(
#     images_dir="images/examples",
#     output_dir="data/processed"
# )
# print(f"Total de filas: {len(df_consolidado)}")
# display(df_consolidado.head(20))

## 7. Guardar resultados

In [ ]:
if 'df' in dir() and not df.empty:
    output_path = ROOT / "data" / "processed" / IMAGEN_PRUEBA.replace(".jpg", ".csv").replace(".png", ".csv")
    extractor.save_csv(df, str(output_path))
    print(f"Guardado en: {output_path}")
else:
    print("No hay datos que guardar todavía.")